In [1]:
import sys
from pathlib import Path

# Find project root dynamically
def get_project_root() -> Path:
    try:
        path = Path(__file__).resolve()
        for parent in [path] + list(path.parents):
            if (parent / "requirements.txt").exists() or (parent / "project").exists():
                return parent
    except NameError:
        pass
    path = Path.cwd().resolve()
    for parent in [path] + list(path.parents):
        if (parent / "requirements.txt").exists() or (parent / "project").exists():
            return parent
    return path

ROOT = get_project_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import torch
import torch.nn as nn
import torch.nn.functional as F


In [2]:
n_embd = 192
n_head = 6
n_layer = 6

projection_dim = 128

dropout = 0.1

block_size = 32

In [3]:
class EncoderHead(nn.Module):

    def __init__(self, head_size):
        super().__init__()

        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)

        self.dropout = nn.Dropout(dropout)

    def forward(self, x):

        B, T, C = x.shape

        k = self.key(x)
        q = self.query(x)
        v = self.value(x)

        wei = q @ k.transpose(-2, -1)
        wei = wei * (k.size(-1) ** -0.5)

        # NO CAUSAL MASK HERE

        wei = F.softmax(wei, dim=-1)
        wei = self.dropout(wei)

        out = wei @ v

        return out

In [4]:
class EncoderMultiHeadAttention(nn.Module):

    def __init__(self, num_heads, head_size):
        super().__init__()

        self.heads = nn.ModuleList([
            EncoderHead(head_size)
            for _ in range(num_heads)
        ])

        self.proj = nn.Linear(num_heads * head_size, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):

        out = torch.cat(
            [h(x) for h in self.heads],
            dim=-1
        )

        out = self.proj(out)
        out = self.dropout(out)

        return out

In [5]:
class TextEncoderBlock(nn.Module):

    def __init__(self, n_embd, n_head):
        super().__init__()

        head_size = n_embd // n_head

        # Self Attention
        self.sa = EncoderMultiHeadAttention(
            n_head,
            head_size
        )

        # Feed Forward
        self.ffwd = FeedForward(n_embd)

        # LayerNorms
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):

        # Self Attention
        x = x + self.sa(self.ln1(x))

        # Feed Forward
        x = x + self.ffwd(self.ln2(x))

        return x

In [6]:
class TextEncoder(nn.Module):

        def __init__(
            self,
            vocab_size,
            max_len,
            n_embd=192,
            n_head=6,
            n_layer=6
        ):
            super().__init__()

            self.token_embedding_table = nn.Embedding(
                vocab_size,
                n_embd
            )

            self.position_embedding_table = nn.Embedding(
                max_len,
                n_embd
            )

            self.blocks = nn.Sequential(
                *[
                    TextEncoderBlock(
                        n_embd,
                        n_head
                    )
                    for _ in range(n_layer)
                ]
            )

            self.ln_f = nn.LayerNorm(n_embd)

        def forward(self, tokens):

          B, T = tokens.shape

          tok_emb = self.token_embedding_table(tokens)

          positions = torch.arange(T, device=tokens.device)
          pos_emb = self.position_embedding_table(positions)

          x = tok_emb + pos_emb

          x = self.blocks(x)

          x = self.ln_f(x)

          return x

In [7]:
class FeedForward(nn.Module):

    def __init__(self, n_embd):

        super().__init__()

        # Sequential:
        # runs layers one after another automatically
        self.net = nn.Sequential(

            # expand dimension
            # transformer MLPs are usually 4x wider
            nn.Linear( n_embd, 4 * n_embd),

            # non-linearity
            # without activation functions, entire network becomes only linear algebra
            nn.ReLU(),

            # project back to original dimension
            nn.Linear(4 * n_embd, n_embd),

            # dropout
            nn.Dropout(dropout)
        )

    def forward(self, x):
        return self.net(x)


In [8]:
class ViTHead(nn.Module):

    def __init__(self, head_size):

        super().__init__()

        self.key = nn.Linear(
            n_embd,
            head_size,
            bias=False
        )

        self.query = nn.Linear(
            n_embd,
            head_size,
            bias=False
        )

        self.value = nn.Linear(
            n_embd,
            head_size,
            bias=False
        )

        self.dropout = nn.Dropout(dropout)

    def forward(self, x):

        k = self.key(x)
        q = self.query(x)
        v = self.value(x)

        wei = q @ k.transpose(-2, -1)

        wei = wei * (k.size(-1) ** -0.5)

        wei = F.softmax(
            wei,
            dim=-1
        )

        wei = self.dropout(wei)

        out = wei @ v

        return out

In [9]:
class ViTMultiHeadAttention(nn.Module):

    def __init__(
        self,
        num_heads,
        head_size
    ):

        super().__init__()

        self.heads = nn.ModuleList([
            ViTHead(head_size)
            for _ in range(num_heads)
        ])

        self.proj = nn.Linear(
            num_heads * head_size,
            n_embd
        )

        self.dropout = nn.Dropout(dropout)

    def forward(self, x):

        out = torch.cat(
            [h(x) for h in self.heads],
            dim=-1
        )

        out = self.proj(out)

        out = self.dropout(out)

        return out


In [10]:
class ViTBlock(nn.Module):

    def __init__(
        self,
        n_embd,
        n_head
    ):

        super().__init__()

        head_size = n_embd // n_head

        self.sa = ViTMultiHeadAttention(
            n_head,
            head_size
        )

        self.ffwd = FeedForward(n_embd)

        self.ln1 = nn.LayerNorm(n_embd)

        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):

        x = x + self.sa(
            self.ln1(x)
        )

        x = x + self.ffwd(
            self.ln2(x)
        )

        return x

In [11]:
class ViTEncoder(nn.Module):

    def __init__(
        self,
        img_size=32,
        patch_size=4,
        in_chans=3,
        n_embd=192,
        n_head=6,
        n_layer=6,
        dropout=0.1
    ):

        super().__init__()

        # NUMBER OF PATCHES
        num_patches = (img_size // patch_size) ** 2

        # PATCH EMBEDDING

        self.patch_embed = nn.Conv2d(
            in_channels=in_chans,
            out_channels=n_embd,
            kernel_size=patch_size,
            stride=patch_size
        )

        # CLS TOKEN

        self.cls_token = nn.Parameter(torch.zeros(1, 1, n_embd))

        # POSITION EMBEDDINGS

        self.pos_embed = nn.Parameter(torch.zeros(1, num_patches + 1, n_embd))

        self.dropout = nn.Dropout(dropout)

        # TRANSFORMER BLOCKS

        self.blocks = nn.Sequential(
            *[
                ViTBlock(n_embd, n_head)
                for _ in range(n_layer)
            ]
        )

        self.norm = nn.LayerNorm(n_embd)

        nn.init.trunc_normal_(self.cls_token, std=0.02)

        nn.init.trunc_normal_(self.pos_embed, std=0.02)

    def forward(self, x):

        B = x.shape[0]

        # PATCH EMBEDDING

        x = self.patch_embed(x)

        # (B,192,8,8)

        x = x.flatten(2)

        # (B,192,64)

        x = x.transpose(1, 2)

        # (B,64,192)

        # CLS TOKEN

        cls = self.cls_token.expand(B, -1, -1)

        x = torch.cat([cls, x], dim=1)

        # (B,65,192)

        # POSITION EMBEDDINGS

        x = x + self.pos_embed

        x = self.dropout(x)

        # TRANSFORMER

        x = self.blocks(x)

        x = self.norm(x)

        # IMPORTANT:
        # return ALL image tokens

        return x

In [12]:
class InfoNCELoss(nn.Module):
    def __init__(self, init_temperature=0.07):
        """
        Args:
            init_temperature: Initial value of tau.
                              CLIP initializes tau around 0.07.
        """
        super().__init__()

        # Learn log(1/tau) instead of tau itself.
        # This guarantees tau always remains positive after exponentiation.
        self.log_inv_tau = nn.Parameter(
            torch.tensor([1.0 / init_temperature]).log()
        )

    def forward(self, image_embeds, text_embeds):
        """
        Args:
            image_embeds : (B, D)
            text_embeds  : (B, D)

        Returns:
            Symmetric InfoNCE loss.
        """

        # Step 1: L2-normalize embeddings
        # Makes dot product equal cosine similarity.
        image_embeds = F.normalize(image_embeds, dim=-1)
        text_embeds = F.normalize(text_embeds, dim=-1)

        # Step 2: Compute inverse temperature
        # Clamp log_inv_tau for numerical stability.
        # log(100) ≈ 4.6052
        log_inv_tau = self.log_inv_tau.clamp(0, 4.6052)
        inv_tau = log_inv_tau.exp()

        # Step 3: Similarity matrix
        # image_embeds : (B, D)
        # text_embeds.T: (D, B)
        # logits       : (B, B)
        # logits[i][j] =
        # cosine_similarity(image_i, text_j) / tau
        logits = inv_tau * (image_embeds @ text_embeds.T)

        # Step 4: Ground-truth labels
        # Correct pairs lie along the diagonal.
        # labels = [0,1,2,...,B-1]
        batch_size = image_embeds.size(0)
        labels = torch.arange(batch_size, device=logits.device)

        # Step 5: Image -> Text loss
        # Each image should retrieve its matching caption.
        loss_i2t = F.cross_entropy(logits, labels)

        # Step 6: Text -> Image loss
        # Each caption should retrieve its matching image.
        loss_t2i = F.cross_entropy(logits.T, labels)

        # Step 7: Symmetric CLIP loss
        loss = (loss_i2t + loss_t2i) / 2

        return loss

In [13]:
class CLIPStyleModel(nn.Module):
    def __init__(
        self,
        vit_encoder,
        text_encoder,
        embed_dim=192,
        projection_dim=128,
        init_temperature=0.07
    ):
        super().__init__()

        # Encoders
        self.vit = vit_encoder
        self.text = text_encoder

        # Projection heads
        self.image_proj = nn.Linear(
            embed_dim,
            projection_dim,
            bias=False
        )

        self.text_proj = nn.Linear(
            embed_dim,
            projection_dim,
            bias=False
        )

        # Contrastive loss
        self.loss_fn = InfoNCELoss(init_temperature)

    def encode_image(self, images):
        """
        Args:
            images: (B, 3, H, W)

        Returns:
            Image embeddings of shape (B, projection_dim)
        """

        # ViT output
        features = self.vit(images)          # (B, 65, 192)

        # CLS token
        cls = features[:, 0]                 # (B, 192)

        # Projection head
        image_embeds = self.image_proj(cls)  # (B, 128)

        return image_embeds

    def encode_text(self, text_tokens, text_mask=None):

        features = self.text(text_tokens)      # (B,T,192)

        if text_mask is not None:

            mask = text_mask.unsqueeze(-1).float()

            pooled = (
                features * mask
            ).sum(dim=1) / mask.sum(dim=1).clamp(min=1e-6)

        else:

            pooled = features.mean(dim=1)

        text_embeds = self.text_proj(pooled)

        return text_embeds

    def forward(self, images, text_tokens, text_mask=None):

        img_e = self.encode_image(images)

        txt_e = self.encode_text(
            text_tokens,
            text_mask
        )

        loss = self.loss_fn(
            img_e,
            txt_e
        )

        return loss, img_e, txt_e

In [14]:
vocab_size = 1000

vit = ViTEncoder(
    img_size=32,
    patch_size=4,
    in_chans=3,
    n_embd=192,
    n_head=6,
    n_layer=6,
    dropout=0.1
)

text = TextEncoder(
    vocab_size=vocab_size,
    max_len=32,
    n_embd=192,
    n_head=6,
    n_layer=6
)

model = CLIPStyleModel(vit, text)

images = torch.randn(8, 3, 32, 32)
tokens = torch.randint(0, vocab_size, (8, 32))
mask = torch.ones(8, 32)

loss, img_e, txt_e = model(images, tokens, mask)

print(loss.shape)
print(img_e.shape)
print(txt_e.shape)

loss.backward()

torch.Size([])
torch.Size([8, 128])
torch.Size([8, 128])


In [16]:
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Total Parameters      : {total_params:,}")
print(f"Trainable Parameters  : {trainable_params:,}")
print(f"Model Size            : {total_params/1e6:.3f} M")

Total Parameters      : 5,601,601
Trainable Parameters  : 5,601,601
Model Size            : 5.602 M


In [15]:
loss, img_e, txt_e = model(images, tokens, mask)

loss.backward()

print(model.image_proj.weight.grad is not None)
print(model.text_proj.weight.grad is not None)
print(model.loss_fn.log_inv_tau.grad is not None)

True
True
True
